In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer 
from sklearn.model_selection import train_test_split,RandomizedSearchCV,KFold,cross_val_score
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor,plot_importance
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

In [2]:
DATA_PATH=Path('data/Features_AB_NYC_2019.csv')
df=pd.read_csv(DATA_PATH)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47692 entries, 0 to 47691
Data columns (total 30 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   neighbourhood                      47692 non-null  str    
 1   latitude                           47692 non-null  float64
 2   longitude                          47692 non-null  float64
 3   price                              47692 non-null  int64  
 4   minimum_nights                     47692 non-null  int64  
 5   number_of_reviews                  47692 non-null  int64  
 6   reviews_per_month                  47692 non-null  float64
 7   calculated_host_listings_count     47692 non-null  int64  
 8   availability_365                   47692 non-null  int64  
 9   days_since_last_review             47692 non-null  float64
 10  has_reviews                        47692 non-null  int64  
 11  dist_to_manhattan_center           47692 non-null  float64
 12  d

In [3]:
df.head(5)

,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,days_since_last_review,...,neighbourhood_group_Queens,neighbourhood_group_Staten Island,popularity_index,reviews_per_listing,name_length,name_word_count,kw_luxury,kw_cozy,kw_view,kw_central
0,Kensington,40.64749,-73.97237,149,1,9,0.21,6,365,262.0,...,0.0,0.0,0.483543,1.5,34,8,0,0,0,0
1,Midtown,40.75362,-73.98377,225,1,45,0.38,2,355,48.0,...,0.0,0.0,1.454884,22.5,21,3,0,0,0,1
2,Harlem,40.80902,-73.94190,150,3,0,0.00,1,365,-1.0,...,0.0,0.0,0.000000,0.0,35,6,0,0,0,0
3,Clinton Hill,40.68514,-73.95976,89,1,270,4.64,1,194,3.0,...,0.0,0.0,25.993831,270.0,31,5,0,1,0,0
4,East Harlem,40.79851,-73.94399,80,10,9,0.10,1,0,231.0,...,0.0,0.0,0.230259,9.0,48,7,0,0,0,1


In [4]:
X=df.drop('price',axis=1)
y=np.log1p(df['price'])

In [5]:
X

,neighbourhood,latitude,longitude,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,days_since_last_review,has_reviews,...,neighbourhood_group_Queens,neighbourhood_group_Staten Island,popularity_index,reviews_per_listing,name_length,name_word_count,kw_luxury,kw_cozy,kw_view,kw_central
0,Kensington,40.64749,-73.97237,1,9,0.21,6,365,262.0,1,...,0.0,0.0,0.483543,1.5,34,8,0,0,0,0
1,Midtown,40.75362,-73.98377,1,45,0.38,2,355,48.0,1,...,0.0,0.0,1.454884,22.5,21,3,0,0,0,1
2,Harlem,40.80902,-73.94190,3,0,0.00,1,365,-1.0,0,...,0.0,0.0,0.000000,0.0,35,6,0,0,0,0
3,Clinton Hill,40.68514,-73.95976,1,270,4.64,1,194,3.0,1,...,0.0,0.0,25.993831,270.0,31,5,0,1,0,0
4,East Harlem,40.79851,-73.94399,10,9,0.10,1,0,231.0,1,...,0.0,0.0,0.230259,9.0,48,7,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47687,Bedford-Stuyvesant,40.67853,-73.94995,2,0,0.00,2,9,-1.0,0,...,0.0,0.0,0.000000,0.0,47,7,0,1,0,0
47688,Bushwick,40.70184,-73.93317,4,0,0.00,2,36,-1.0,0,...,0.0,0.0,0.000000,0.0,45,5,0,0,0,0
47689,Harlem,40.81475,-73.94867,10,0,0.00,1,27,-1.0,0,...,0.0,0.0,0.000000,0.0,39,5,0,0,0,0
47690,Hell's Kitchen,40.75751,-73.99112,1,0,0.00,6,2,-1.0,0,...,0.0,0.0,0.000000,0.0,36,6,0,1,0,0


In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
categorical_te=['neighbourhood']

In [8]:
df.columns.to_list()

['neighbourhood',
 'latitude',
 'longitude',
 'price',
 'minimum_nights',
 'number_of_reviews',
 'reviews_per_month',
 'calculated_host_listings_count',
 'availability_365',
 'days_since_last_review',
 'has_reviews',
 'dist_to_manhattan_center',
 'dist_to_jfk_km',
 'dist_to_brooklyn_bridge',
 'room_type_Entire home/apt',
 'room_type_Private room',
 'room_type_Shared room',
 'neighbourhood_group_Bronx',
 'neighbourhood_group_Brooklyn',
 'neighbourhood_group_Manhattan',
 'neighbourhood_group_Queens',
 'neighbourhood_group_Staten Island',
 'popularity_index',
 'reviews_per_listing',
 'name_length',
 'name_word_count',
 'kw_luxury',
 'kw_cozy',
 'kw_view',
 'kw_central']

In [9]:
numeric_features = [
    'latitude', 'longitude', 'minimum_nights', 'number_of_reviews',
    'reviews_per_month', 'calculated_host_listings_count', 'availability_365',
    'days_since_last_review', 'dist_to_manhattan_center', 'dist_to_jfk_km',
    'dist_to_brooklyn_bridge', 'popularity_index', 'reviews_per_listing',
    'name_length', 'name_word_count'
]

In [10]:
passthrough_features = [
    'has_reviews', 'kw_luxury', 'kw_cozy', 'kw_view', 'kw_central',
    'room_type_Entire home/apt', 'room_type_Private room', 'room_type_Shared room',
    'neighbourhood_group_Bronx', 'neighbourhood_group_Brooklyn',
    'neighbourhood_group_Manhattan', 'neighbourhood_group_Queens',
    'neighbourhood_group_Staten Island'
]

In [11]:
preprocessor_tree = ColumnTransformer(
    transformers=[('te', TargetEncoder(smoothing=10), categorical_te),],
    remainder='passthrough'  
)

In [12]:
preprocessor_linear = ColumnTransformer(
    transformers=[('te', TargetEncoder(smoothing=10), categorical_te),('scale', StandardScaler(), numeric_features),],remainder='passthrough')

### Define Pipelines

In [13]:
ridge_pipe=Pipeline([('preprocessor',preprocessor_linear),('model',Ridge(alpha=1.0)),])

In [14]:
rf_pipe=Pipeline([('preprocessor',preprocessor_tree),('model',RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1)),])

In [15]:
xgb_pipe=Pipeline([('preprocessor',preprocessor_tree),('model',XGBRegressor(objective='reg:squarederror',n_estimators=1000,seed=42))])

### KFold cross-validation

In [16]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_ridge = cross_val_score(ridge_pipe,X_train,y_train,cv=kf,scoring='neg_root_mean_squared_error',n_jobs=-1)
scores_rf = cross_val_score(rf_pipe,X_train,y_train,cv=kf,scoring='neg_root_mean_squared_error',n_jobs=-1)
scores_xgb = cross_val_score(xgb_pipe,X_train,y_train,cv=kf,scoring='neg_root_mean_squared_error',n_jobs=-1)

In [17]:
rmse_ridge = -scores_ridge
rmse_rf = -scores_rf
rmse_xgb = -scores_xgb

In [18]:
print(f"Ridge RMSE: {rmse_ridge.mean():.4f} ± {rmse_ridge.std():.4f}")
print(f"RF RMSE: {rmse_rf.mean():.4f} ± {rmse_rf.std():.4f}")
print(f"XGB RMSE: {rmse_xgb.mean():.4f} ± {rmse_xgb.std():.4f}")

Ridge RMSE: 0.4070 ± 0.0032
RF RMSE: 0.3861 ± 0.0033
XGB RMSE: 0.4092 ± 0.0041


In [19]:
rf_pipe.fit(X_train, y_train)
y_pred_usd = np.expm1(rf_pipe.predict(X_test))
y_test_usd = np.expm1(y_test)
print(f"Test RMSE: ${root_mean_squared_error(y_test_usd, y_pred_usd):.2f}")
print(f"Test MAE:  ${mean_absolute_error(y_test_usd, y_pred_usd):.2f}")

Test RMSE: $74.83
Test MAE:  $41.98
